# 淘宝用户购物行为分析

## 概述

本数据集包含2017年11月25日至2017年12月3日期间约一百万名随机用户的行为记录，包括浏览、收藏、加购和购买。每行由用户ID、商品ID、商品类目ID、行为类型和时间戳组成。

## 字段说明

| 字段 | 含义 |
|---|---|
| 用户ID | 序列化后的用户ID |
| 商品ID | 序列化后的商品ID |
| 商品类目ID | 序列化后的商品所属类目ID |
| 行为类型 | `pv`、`buy`、`cart`、`fav` |
| 时间戳 | 行为发生的Unix时间戳 |

`pv` 表示浏览商品详情页，`buy` 表示购买，`cart` 表示加入购物车，`fav` 表示收藏。

## 数据规模（数据集说明口径）

| 维度 | 数量 |
|---|---:|
| 用户数量 | 987,994 |
| 商品数量 | 4,162,024 |
| 商品类目数量 | 9,439 |
| 行为数量 | 100,150,807 |

## 重要限制

- 数据仅覆盖9天，适合描述该窗口内的行为，不代表长期用户生命周期。
- 数据不含价格、订单金额和利润，不能直接推断GMV、收入、利润或LTV。
- 本Notebook中的“意向”指收藏或加购；严格漏斗要求同一用户对同一商品依次发生浏览、意向和购买。
- 修改后的Notebook已清除旧输出，需要在具备原始数据的本地环境中从头执行后再引用数值结论。

## 引用

1. Han Z, Xiang L, Pengye Z, et al. 2018. Learning Tree-based Deep Model for Recommender Systems. KDD.
2. Han Z, Daqing C, Ziru X, et al. 2019. Joint Optimization of Tree-based Index and Deep Model for Recommender Systems. NeurIPS.
3. Jingwei Z, Ziru X, Wei D, et al. 2020. Learning Optimal Tree Models under Beam Search. ICML.

## AI使用情况

- 使用Gemini辅助代码纠错、文案编写与可视化代码调试。
- 所有指标定义、计算结果和业务结论仍需由作者复核。

# 分析目标

1. 描述分析期内的整体行为漏斗与分时段行为特征。
2. 根据流量和严格浏览→意向→购买漏斗表现，对商品及类目进行分层。
3. 在缺少金额字段的前提下，使用行为特征和K-Means识别相对高活跃、高购买用户群体。
4. 将可验证的数据发现与待验证的业务解释、运营建议明确分开。

# 初始化与数据导入

In [ ]:
# 导入分析依赖并声明输入
import duckdb
import numpy as np
import pandas as pd
from plotly import graph_objects as go
import plotly.io as pio
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

pio.renderers.default = "notebook"

file_name = "UserBehavior.csv"
db = "UserBehavior.db"
analysis_timezone = "Asia/Shanghai"
analysis_start_ts = 1511539200       # 2017-11-25 00:00:00 Asia/Shanghai
analysis_end_ts_exclusive = 1512316800  # 2017-12-04 00:00:00 Asia/Shanghai
random_state = 50

In [ ]:
# 创建原始数据表
connection = duckdb.connect(db)
connection.execute(f"""
    CREATE OR REPLACE TABLE Raw AS
    SELECT
        column0 AS user_id,
        column1 AS good_id,
        column2 AS prop_id,
        column3 AS act,
        column4 AS ts
    FROM read_csv('{file_name}');
""")
connection.close()
print("数据库创建成功！")

In [ ]:
#检查数据
_ = duckdb.connect(db)
print(_.execute("SELECT * FROM Raw LIMIT 5;").fetchdf())
#print(_.execute('SUMMARIZE Raw;').fetchdf())
_.close()

---

# 数据清洗

因源文件数据量过大，将database数据导入到dataframe再使用pandas进行数据清洗效率将十分低下，故选择直接在database中进行数据清洗。

In [ ]:
connection = duckdb.connect(db)

# 去重
connection.execute("""
    CREATE OR REPLACE TABLE Cleaned AS
    SELECT DISTINCT * FROM Raw;
""")

# 删除关键字段缺失、行为类型非法或不在完整分析窗口内的记录
connection.execute(f"""
    DELETE FROM Cleaned
    WHERE user_id IS NULL
       OR good_id IS NULL
       OR prop_id IS NULL
       OR act IS NULL
       OR ts IS NULL
       OR act NOT IN ('pv', 'cart', 'fav', 'buy')
       OR ts < {analysis_start_ts}
       OR ts >= {analysis_end_ts_exclusive};
""")

# 仅保留分析期内至少有一次浏览行为的用户
connection.execute("""
    DELETE FROM Cleaned
    WHERE user_id NOT IN (
        SELECT DISTINCT user_id FROM Cleaned WHERE act = 'pv'
    );
""")

print("数据清洗完成！")
connection.close()

In [ ]:
#检查数据
_ = duckdb.connect(db)
print(_.execute("SELECT * FROM Cleaned LIMIT 5;").fetchdf())
#print(_.execute('SUMMARIZE Cleaned;').fetchdf())
_.close()

---

# 数据格式化

In [ ]:
connection = duckdb.connect(db)
connection.execute(f"SET TimeZone = '{analysis_timezone}';")
connection.execute("""
CREATE OR REPLACE VIEW vCleaned AS
WITH formatted AS (
    SELECT
        *,
        to_timestamp(ts)::TIMESTAMPTZ AS dt
    FROM Cleaned
)
SELECT
    user_id,
    good_id,
    prop_id,
    act,
    dt AS 时间,
    EXTRACT(DAY FROM dt) AS 日期,
    EXTRACT(HOUR FROM dt) AS 小时,
    EXTRACT(ISODOW FROM dt) AS 星期
FROM formatted;
""")
connection.close()

In [ ]:
#检查数据
_ = duckdb.connect(db)
print(_.execute("SELECT * FROM vCleaned LIMIT 5;").fetchdf())
_.close()

---

# 宏观数据分析

In [ ]:
# 构建整体行为漏斗指标
connection = duckdb.connect(db)
connection.execute("""
    CREATE OR REPLACE VIEW macro AS
    WITH totals AS (
        SELECT
            COUNT(CASE WHEN act = 'pv' THEN 1 END) AS 总浏览次数,
            COUNT(CASE WHEN act = 'fav' THEN 1 END) AS 总收藏次数,
            COUNT(CASE WHEN act = 'cart' THEN 1 END) AS 总购物车次数,
            COUNT(CASE WHEN act IN ('fav', 'cart') THEN 1 END) AS 总意向次数,
            COUNT(CASE WHEN act = 'buy' THEN 1 END) AS 总购买次数
        FROM Cleaned
    )
    SELECT
        *,
        ROUND(总收藏次数 * 100.0 / NULLIF(总浏览次数, 0), 2) AS 浏览转收藏比,
        ROUND(总购物车次数 * 100.0 / NULLIF(总浏览次数, 0), 2) AS 浏览转购物车比,
        ROUND(总购买次数 * 100.0 / NULLIF(总意向次数, 0), 2) AS 购买意向行为比,
        ROUND(总购买次数 * 100.0 / NULLIF(总浏览次数, 0), 2) AS 浏览购买行为比
    FROM totals;
""")
connection.close()

In [ ]:
#检查数据
_ = duckdb.connect(db)
print(_.execute("SELECT * FROM macro LIMIT 5;").fetchdf())
_.close()

In [ ]:
# 读取整体行为量用于展示；该图是阶段行为量对比，不代表严格顺序漏斗
connection = duckdb.connect(db)
df = connection.execute(
    "SELECT 总浏览次数, 总意向次数, 总购买次数 FROM macro"
).fetchdf()
connection.close()

In [ ]:
fig = go.Figure(go.Funnel(
    y=["浏览行为", "收藏或加购行为", "购买行为"],
    x=df.iloc[0].to_numpy(),
    textinfo="value+percent total",
    marker_color=["#4C78A8", "#F58518", "#E45756"],
    opacity=0.75,
))
fig.update_layout(title="分析期内各阶段行为量（非严格顺序漏斗）")
fig.show()

浏览行为量明显高于收藏、加购和购买行为量，说明浏览后的意向形成是值得进一步诊断的环节。但这里统计的是各类行为的总次数，并未验证同一用户是否按“浏览→收藏/加购→购买”的顺序转化，因此不能把阶段行为量之比解释为因果转化率。

若浏览→意向比例提高1个百分点，按浏览基数可计算相应的新增意向行为量；新增订单和GMV仍取决于后续的真实增量转化、客单价和优惠成本。本数据缺少金额字段，也没有实验对照，因此不在此直接估算营业额。

可测试的优化方向包括提高推荐相关性、降低商品信息理解成本，以及针对已经产生浏览或意向行为的用户设计分层触达。上述策略应通过A/B测试验证增量订单、毛利和用户体验，而不是仅依据描述性相关关系上线。

In [ ]:
#构建时间序列数据库
_ = duckdb.connect(db)

_.execute("""
    CREATE OR REPLACE VIEW Tmacro AS
        SELECT
            小时,
            星期,
            count(CASE WHEN act='pv' THEN user_id END) AS 总浏览次数,
            count(CASE WHEN act='fav' THEN user_id END) AS 总收藏次数,
            count(CASE WHEN act='cart' THEN user_id END) AS 总购物车次数,
            count(CASE WHEN act='buy' THEN user_id END) AS 总购买次数
        FROM vCleaned
        GROUP BY 1,2
""")

_.close()

In [ ]:
#检查数据
_ = duckdb.connect(db)
print(_.execute("SELECT * FROM Tmacro LIMIT 5;").fetchdf())
_.close()

In [ ]:
#读取绘制图所需数据
_ = duckdb.connect(db)
df = _.execute("""
SELECT 
小时,
SUM(总浏览次数) AS 总浏览次数,
SUM(总收藏次数) AS 总收藏次数,
SUM(总购物车次数) AS 总购物车次数,
SUM(总购买次数) AS 总购买次数
FROM Tmacro GROUP BY 小时""").fetchdf()
_.close()

In [ ]:
#按小时的互动直方图（包含浏览次数）
n = df['小时']
fig = go.Figure(data=[
    go.Bar(name='浏览次数', x=n, y=df['总浏览次数']),
    go.Bar(name='收藏次数', x=n, y=df['总收藏次数']),
    go.Bar(name='购物车次数', x=n, y=df['总购物车次数']),
    go.Bar(name='购买次数', x=n, y=df['总购买次数'])
])
fig.update_layout(barmode='stack',title='<b>按小时的互动直方图（包含浏览次数）</b>')
fig.show()
#按小时的互动直方图（不包含浏览次数）
n = df['小时']
fig = go.Figure(data=[
    go.Bar(name='收藏次数', x=n, y=df['总收藏次数']),
    go.Bar(name='购物车次数', x=n, y=df['总购物车次数']),
    go.Bar(name='购买次数', x=n, y=df['总购买次数'])
])
fig.update_layout(barmode='stack',title='<b>按小时的互动直方图（不包含浏览次数）</b>')
fig.show()

In [ ]:
# 按小时计算正确的浏览购买行为比
connection = duckdb.connect(db)
df = connection.execute("""
SELECT
    小时,
    SUM(总浏览次数) AS 总浏览次数,
    SUM(总购买次数) AS 总购买次数,
    ROUND(
        SUM(总购买次数) * 100.0 / NULLIF(SUM(总浏览次数), 0),
        2
    ) AS 浏览购买行为比
FROM Tmacro
GROUP BY 小时
ORDER BY 小时;
""").fetchdf()
connection.close()

In [ ]:
# 展示各小时浏览购买行为比
fig = go.Figure(data=[
    go.Bar(
        name="浏览购买行为比",
        x=df["小时"],
        y=df["浏览购买行为比"],
        marker=dict(color=df["浏览购买行为比"], coloraxis="coloraxis"),
    )
])

weighted_rate = (
    df["总购买次数"].sum() * 100.0 /
    df["总浏览次数"].sum()
)
fig.add_hline(
    y=weighted_rate,
    line_dash="dash",
    line_color="grey",
    annotation_text=f"全时段加权比例：{weighted_rate:.2f}%",
    annotation_position="bottom left",
)
fig.update_layout(
    title="各小时浏览购买行为比（购买行为数/浏览行为数）",
    xaxis_title="小时",
    yaxis_title="比例（%）",
    yaxis_rangemode="tozero",
)
fig.show()

分时段图可用于识别行为量和浏览购买行为比的时段差异，但不能仅凭这些汇总数据推断用户心理或营销效果。不同小时的差异还可能来自用户构成、商品类目、活动安排、星期结构及跨时段购买等因素。

后续应按日期、星期、商品类目和用户群体进行分层，并检查结论是否稳定。若准备调整推送时间或晚间优惠，应通过随机对照实验评估增量购买、毛利、退订率和用户打扰等指标。

---

# 商品价值分析

商品与商品类目分析使用严格的三阶段漏斗：同一用户对同一商品（或类目）先浏览，之后收藏或加购，再之后购买。这样可以保证浏览人数≥意向人数≥购买人数，避免出现超过100%的“转化率”。

先创建未过滤的基础漏斗，再从基础漏斗计算浏览人数的5%分位数，最后过滤低于阈值的实体。由于5%分位数可能很低，输出中会同时展示实际阈值，便于判断过滤是否有实质意义。

In [ ]:
# 创建未过滤的严格商品/类目漏斗，再计算流量阈值
connection = duckdb.connect(db)

def create_strict_funnel_base(view_name, entity_column):
    connection.execute(f"""
    CREATE OR REPLACE VIEW {view_name} AS
    WITH first_pv AS (
        SELECT
            user_id,
            {entity_column} AS entity_id,
            MIN(ts) AS first_pv_ts
        FROM Cleaned
        WHERE act = 'pv'
        GROUP BY user_id, {entity_column}
    ), first_intent AS (
        SELECT
            first_pv.user_id,
            first_pv.entity_id,
            first_pv.first_pv_ts,
            MIN(events.ts) AS first_intent_ts
        FROM first_pv
        LEFT JOIN Cleaned AS events
          ON events.user_id = first_pv.user_id
         AND events.{entity_column} = first_pv.entity_id
         AND events.act IN ('fav', 'cart')
         AND events.ts > first_pv.first_pv_ts
        GROUP BY first_pv.user_id, first_pv.entity_id, first_pv.first_pv_ts
    ), first_buy AS (
        SELECT
            first_intent.user_id,
            first_intent.entity_id,
            first_intent.first_pv_ts,
            first_intent.first_intent_ts,
            MIN(events.ts) AS first_buy_ts
        FROM first_intent
        LEFT JOIN Cleaned AS events
          ON events.user_id = first_intent.user_id
         AND events.{entity_column} = first_intent.entity_id
         AND events.act = 'buy'
         AND events.ts > first_intent.first_intent_ts
        GROUP BY
            first_intent.user_id,
            first_intent.entity_id,
            first_intent.first_pv_ts,
            first_intent.first_intent_ts
    )
    SELECT
        entity_id,
        COUNT(*) AS pv_cnt,
        COUNT(first_intent_ts) AS intent_cnt,
        COUNT(first_buy_ts) AS buy_cnt
    FROM first_buy
    GROUP BY entity_id;
    """)

create_strict_funnel_base("Good_Funnel_Base", "good_id")
create_strict_funnel_base("Prop_Funnel_Base", "prop_id")

good_pv_threshold = connection.execute(
    "SELECT quantile_cont(pv_cnt, 0.05) FROM Good_Funnel_Base"
).fetchone()[0]
prop_pv_threshold = connection.execute(
    "SELECT quantile_cont(pv_cnt, 0.05) FROM Prop_Funnel_Base"
).fetchone()[0]

print(f"商品浏览用户数5%分位数：{good_pv_threshold:.2f}")
print(f"类目浏览用户数5%分位数：{prop_pv_threshold:.2f}")
connection.close()

In [ ]:
# 根据已计算的阈值创建最终严格漏斗视图
connection = duckdb.connect(db)

connection.execute(f"""
CREATE OR REPLACE VIEW Good_Funnel AS
SELECT
    good_id AS 商品id,
    pv_cnt AS 独立浏览人数,
    intent_cnt AS 独立意向人数,
    buy_cnt AS 独立购买人数,
    ROUND(intent_cnt * 100.0 / NULLIF(pv_cnt, 0), 2) AS 浏览转意向率,
    ROUND(buy_cnt * 100.0 / NULLIF(intent_cnt, 0), 2) AS 意向转购买率,
    ROUND(buy_cnt * 100.0 / NULLIF(pv_cnt, 0), 2) AS 浏览转购买率
FROM Good_Funnel_Base
WHERE pv_cnt >= {good_pv_threshold};
""")

connection.execute(f"""
CREATE OR REPLACE VIEW Prop_Funnel AS
SELECT
    prop_id AS 商品类目id,
    pv_cnt AS 独立浏览人数,
    intent_cnt AS 独立意向人数,
    buy_cnt AS 独立购买人数,
    ROUND(intent_cnt * 100.0 / NULLIF(pv_cnt, 0), 2) AS 浏览转意向率,
    ROUND(buy_cnt * 100.0 / NULLIF(intent_cnt, 0), 2) AS 意向转购买率,
    ROUND(buy_cnt * 100.0 / NULLIF(pv_cnt, 0), 2) AS 浏览转购买率
FROM Prop_Funnel_Base
WHERE pv_cnt >= {prop_pv_threshold};
""")

connection.close()

In [ ]:
# 检查严格漏斗结果
connection = duckdb.connect(db)
print(connection.execute(
    "SELECT * FROM Good_Funnel ORDER BY 浏览转购买率 DESC LIMIT 5"
).fetchdf())
print(connection.execute(
    "SELECT * FROM Prop_Funnel ORDER BY 浏览转购买率 DESC LIMIT 5"
).fetchdf())
connection.close()

使用独立浏览人数和严格浏览转购买率进行四象限分层。考虑到行为数据通常右偏，使用中位数而不是均值作为分界线。

|类型|浏览人数|浏览转购买率|解释|
|---|---|---|---|
|现象型|高|高|兼具流量与转化表现|
|流量型|高|低|有流量但转化表现相对较弱|
|潜力型|低|高|转化表现较好但流量有限|
|长尾型|低|低|当前窗口内流量和转化均较低|

“长尾型”只描述本分析窗口内的相对位置，不等同于商品质量差或应该下架。数据不含销售额、成本和利润，因此不能据此判断利润贡献。

In [ ]:
# 商品类目四象限与汇总
connection = duckdb.connect(db)
prop_df = connection.execute("""
    SELECT 独立浏览人数, 浏览转购买率, 独立购买人数
    FROM Prop_Funnel
""").fetchdf()
connection.close()

prop_pv_cut = prop_df["独立浏览人数"].median()
prop_conversion_cut = prop_df["浏览转购买率"].median()
prop_df["类型"] = np.select(
    [
        (prop_df["独立浏览人数"] >= prop_pv_cut) & (prop_df["浏览转购买率"] >= prop_conversion_cut),
        (prop_df["独立浏览人数"] >= prop_pv_cut) & (prop_df["浏览转购买率"] < prop_conversion_cut),
        (prop_df["独立浏览人数"] < prop_pv_cut) & (prop_df["浏览转购买率"] >= prop_conversion_cut),
    ],
    ["现象型", "流量型", "潜力型"],
    default="长尾型",
)
print(prop_df.groupby("类型").agg(
    类目数=("类型", "size"),
    严格漏斗购买人数合计=("独立购买人数", "sum"),
).sort_values("类目数", ascending=False))

fig = go.Figure(go.Scatter(
    x=prop_df["独立浏览人数"],
    y=prop_df["浏览转购买率"],
    mode="markers",
    text=prop_df["类型"],
    marker=dict(
        size=prop_df["独立购买人数"],
        sizemode="area",
        sizeref=max(prop_df["独立购买人数"].max() / 60**2, 1),
        color=prop_df["浏览转购买率"],
        coloraxis="coloraxis",
        opacity=0.7,
    ),
))
fig.add_vline(x=prop_pv_cut, line_dash="dash", line_color="grey")
fig.add_hline(y=prop_conversion_cut, line_dash="dash", line_color="grey")
fig.update_layout(
    title="商品类目四象限（中位数分界）",
    xaxis_title="独立浏览人数",
    yaxis_title="严格浏览转购买率（%）",
)
fig.show()

In [ ]:
# 商品四象限与汇总
connection = duckdb.connect(db)
good_df = connection.execute("""
    SELECT 独立浏览人数, 浏览转购买率, 独立购买人数
    FROM Good_Funnel
""").fetchdf()
connection.close()

good_pv_cut = good_df["独立浏览人数"].median()
good_conversion_cut = good_df["浏览转购买率"].median()
good_df["类型"] = np.select(
    [
        (good_df["独立浏览人数"] >= good_pv_cut) & (good_df["浏览转购买率"] >= good_conversion_cut),
        (good_df["独立浏览人数"] >= good_pv_cut) & (good_df["浏览转购买率"] < good_conversion_cut),
        (good_df["独立浏览人数"] < good_pv_cut) & (good_df["浏览转购买率"] >= good_conversion_cut),
    ],
    ["现象型", "流量型", "潜力型"],
    default="长尾型",
)
print(good_df.groupby("类型").agg(
    商品数=("类型", "size"),
    严格漏斗购买人数合计=("独立购买人数", "sum"),
).sort_values("商品数", ascending=False))

fig = go.Figure(go.Scatter(
    x=good_df["独立浏览人数"],
    y=good_df["浏览转购买率"],
    mode="markers",
    text=good_df["类型"],
    marker=dict(
        size=good_df["独立购买人数"],
        sizemode="area",
        sizeref=max(good_df["独立购买人数"].max() / 60**2, 1),
        color=good_df["浏览转购买率"],
        coloraxis="coloraxis",
        opacity=0.7,
    ),
))
fig.add_vline(x=good_pv_cut, line_dash="dash", line_color="grey")
fig.add_hline(y=good_conversion_cut, line_dash="dash", line_color="grey")
fig.update_layout(
    title="商品四象限（中位数分界）",
    xaxis_title="独立浏览人数",
    yaxis_title="严格浏览转购买率（%）",
)
fig.show()

四象限汇总表用于比较各类型的实体数量与严格漏斗购买人数贡献。只有在重新执行后，才能依据输出判断哪类商品或类目数量最多。

可将“潜力型”和“流量型”作为后续实验候选：前者测试增加合适场景下的曝光，后者测试详情页、价格呈现或推荐人群优化。四象限是相对分类，不构成商品质量、利润或下架决策的充分证据；上线策略前还需补充价格、毛利、退款、库存和商家质量指标。

# 用户价值分析

数据不包含订单金额，因此无法构建完整RFM中的金额指标，也不能估算LTV。本节使用浏览商品数、真实活跃天数、最近活跃间隔和购买次数进行K-Means行为分群。

聚类仅描述本分析窗口内用户之间的相对差异。“高价值用户”在此应理解为“高活跃、高购买行为用户”，而不是已验证的高收入或高利润用户。

In [ ]:
# 创建用户行为特征视图
connection = duckdb.connect(db)
connection.execute(f"SET TimeZone = '{analysis_timezone}';")
connection.execute("""
CREATE OR REPLACE VIEW active_user AS
SELECT
    user_id,
    COUNT(*) AS total_actions,
    COUNT(DISTINCT good_id) AS unique_goods_browsed,
    COUNT(CASE WHEN act = 'buy' THEN 1 END) AS total_buy,
    COUNT(DISTINCT CAST(时间 AS DATE)) AS active_days,
    DATEDIFF(
        'day',
        MAX(CAST(时间 AS DATE)),
        DATE '2017-12-03'
    ) AS recency_days
FROM vCleaned
GROUP BY user_id;
""")
connection.close()

In [ ]:
# 构建聚类数据集
connection = duckdb.connect(db)
df = connection.execute("""
SELECT
    total_actions AS 总行为次数,
    total_buy AS 总购买次数,
    unique_goods_browsed AS 浏览商品数,
    active_days AS 活跃天数,
    recency_days AS 最近活跃间隔
FROM active_user;
""").fetchdf()
connection.close()

In [ ]:
# 评估聚类数量，并以三个可解释群体完成最终聚类
feature_columns = ["浏览商品数", "活跃天数", "最近活跃间隔", "总购买次数"]
df[feature_columns] = df[feature_columns].fillna(0).clip(lower=0)

# 计数特征通常右偏，log1p可降低极端用户对距离的支配
log_features = np.log1p(df[feature_columns])
scaler = StandardScaler()
X = scaler.fit_transform(log_features)

# 在固定随机样本上比较K，避免对近百万用户直接计算完整轮廓系数
sample_size = min(20_000, len(df))
sample_index = df.sample(n=sample_size, random_state=random_state).index
X_sample = X[sample_index]

silhouette_results = {}
for candidate_k in range(2, 7):
    candidate_model = KMeans(
        n_clusters=candidate_k,
        random_state=random_state,
        n_init=10,
    )
    candidate_labels = candidate_model.fit_predict(X_sample)
    silhouette_results[candidate_k] = silhouette_score(
        X_sample,
        candidate_labels,
        sample_size=min(5_000, sample_size),
        random_state=random_state,
    )

print("候选K的样本轮廓系数：", silhouette_results)
print("轮廓系数最佳K：", max(silhouette_results, key=silhouette_results.get))

# 保留三个群体，便于形成高、常规、低三档；同时公开最佳K供复核
selected_k = 3
kmeans = KMeans(
    n_clusters=selected_k,
    random_state=random_state,
    n_init=10,
)
df["cluster"] = kmeans.fit_predict(X)

# 最近活跃间隔越小越好，其他三个特征越大代表行为价值越高
cluster_value_score = (
    kmeans.cluster_centers_[:, 0]
    + kmeans.cluster_centers_[:, 1]
    - kmeans.cluster_centers_[:, 2]
    + kmeans.cluster_centers_[:, 3]
)
high_cluster = int(np.argmax(cluster_value_score))
low_cluster = int(np.argmin(cluster_value_score))
regular_cluster = next(
    cluster for cluster in range(selected_k)
    if cluster not in {high_cluster, low_cluster}
)
cluster_label_map = {
    high_cluster: "高活跃高购买用户",
    regular_cluster: "常规用户",
    low_cluster: "低活跃低购买用户",
}

cluster_summary = df.groupby("cluster")[feature_columns + ["总行为次数"]].mean()
cluster_summary["用户数"] = df["cluster"].value_counts().reindex(cluster_summary.index)
cluster_summary["用户占比"] = cluster_summary["用户数"] / len(df) * 100
purchase_by_cluster = df.groupby("cluster")["总购买次数"].sum()
cluster_summary["购买行为占比"] = (
    purchase_by_cluster / purchase_by_cluster.sum() * 100
)
cluster_summary["用户类型"] = cluster_summary.index.map(cluster_label_map)
print(cluster_summary.round(2))

本分析固定使用三类以便形成清晰、可运营的分群，同时输出K=2至K=6的样本轮廓系数。如果轮廓系数最佳K不是3，应在模型拟合度与业务可解释性之间作出明确取舍，而不能仅凭主观调参确定分类数。

群体名称依据聚类中心自动映射，避免把聚类编号错误地当成固定业务标签。结论应以重新执行后输出的用户占比和购买行为占比为准。

In [ ]:
# 绘制用户群体气泡图，按cluster索引显式对齐人数和标签
plot_summary = cluster_summary.sort_index()
fig = go.Figure(go.Scatter(
    x=plot_summary["浏览商品数"],
    y=plot_summary["活跃天数"],
    mode="markers+text",
    text=plot_summary["用户类型"],
    textposition="middle center",
    marker=dict(
        size=plot_summary["用户数"].to_numpy(),
        sizemode="area",
        sizeref=max(plot_summary["用户数"].max() / 90**2, 1),
        color=plot_summary["购买行为占比"].to_numpy(),
        coloraxis="coloraxis",
        opacity=0.75,
    ),
))
fig.update_layout(
    title="用户行为分群（气泡大小表示用户数）",
    xaxis_title="平均浏览商品数",
    yaxis_title="平均活跃天数",
)
fig.show()

用户分群的规模、活跃特征和购买行为贡献以重新运行后的汇总表为准。由于数据窗口只有9天，这些标签只表示短期行为差异，不能直接解释为长期客户价值。

特别需要区分“购买行为占比”和“收入占比”：本数据没有金额字段，因此不能声称某一群体掌握近一半收益。若要评估真正的客户价值，需要补充订单金额、毛利、退款、获客成本和更长时间范围的数据。

运营建议（需通过实验验证）：

- 常规用户：测试更清晰的商品信息、购物车提醒和适度的限时权益，重点观察增量购买、毛利和退订率。
- 低活跃低购买用户：优先采用低成本自动化触达，并设置频控，避免补贴和消息打扰超过潜在收益。
- 高活跃高购买用户：测试会员权益、服务体验和流失预警；避免仅依靠大额优惠，以免侵蚀毛利或奖励本来就会购买的用户。

这些建议来自描述性分群，不代表已证明的因果效果。应使用随机对照实验或准实验设计验证。

In [ ]:
# 最终一致性检查：重新执行时，任一断言失败都应阻止发布结论
connection = duckdb.connect(db)

for view_name in ["Good_Funnel", "Prop_Funnel"]:
    invalid_funnel_rows = connection.execute(f"""
        SELECT COUNT(*)
        FROM {view_name}
        WHERE 独立浏览人数 < 独立意向人数
           OR 独立意向人数 < 独立购买人数
           OR 浏览转意向率 NOT BETWEEN 0 AND 100
           OR 意向转购买率 NOT BETWEEN 0 AND 100
           OR 浏览转购买率 NOT BETWEEN 0 AND 100;
    """).fetchone()[0]
    assert invalid_funnel_rows == 0, f"{view_name} 存在无效漏斗记录"

assert np.isclose(cluster_summary["用户占比"].sum(), 100.0)
if df["总购买次数"].sum() > 0:
    assert np.isclose(cluster_summary["购买行为占比"].sum(), 100.0)

connection.close()
print("指标范围、严格漏斗顺序和聚类占比检查通过。")